# FastAPI: маршруты, валидация и тестирование

Практика построена как последовательное развитие приложения:

1. базовые маршруты и параметры;
2. разделение маршрутов через `APIRouter`;
3. Pydantic-схемы и обработка ошибок;
4. контракт текстового inference endpoint;
5. проверка API через `TestClient`.

Для проверки маршрутов используется `TestClient`. Он отправляет запросы напрямую объекту `FastAPI` внутри Python-процесса, поэтому запуск `uvicorn` для выполнения notebook не требуется.

Обозначения `v1`, `v2` и `v3` соответствуют файлам `main_v1.py`, `main_v2.py`, `main_v3.py` и префиксам `/api/v1`, `/api/v2`, `/api/v3`.

| Имя в notebook | Файл приложения | Адреса | Что показывает |
| --- | --- | --- | --- |
| `basic_app` | [`app/main.py`](app/main.py) | `/`, `/items/{item_id}`, `/search` | базовые маршруты и параметры |
| `v1_app` | [`app/main_v1.py`](app/main_v1.py) | `/api/v1/...` | разделение маршрутов через `APIRouter` |
| `v2_app` | [`app/main_v2.py`](app/main_v2.py) | `/api/v2/...` | Pydantic-схемы, валидация и ошибки |
| `v3_app` | [`app/main_v3.py`](app/main_v3.py) | `/health`, `/api/v3/predict-text` | контракт заглушки текстовой модели |


## Как устроены учебные версии

Каждая версия является отдельным объектом `FastAPI` и имеет собственную точку входа. Файл `main_v*.py` создаёт приложение и подключает только роутеры своей версии. Поэтому `TestClient(v1_app)` работает с `v1`, а `TestClient(v2_app)` с `v2`.

```text
app/
  main_v1.py            приложение v1
  main_v2.py            приложение v2
  main_v3.py            приложение v3 и маршрут /health
  api/
    v1/
      users.py           чтение пользователей
      items.py           чтение предметов
    v2/
      users.py           создание и получение пользователей
      items.py           создание и получение предметов
    v3/
      llm.py             заглушка текстовой модели
  models/
    schemas.py           Pydantic-модели для v2 и v3
```

### Версия 1

[`app/main_v1.py`](app/main_v1.py) создаёт приложение и подключает роутеры `users` и `items` с общим префиксом `/api/v1`. Внутри [`users.py`](app/api/v1/users.py) роутер добавляет `/users`, а обработчик добавляет `/{user_id}`. Полный путь складывается так: `/api/v1` + `/users` + `/{user_id}` = `/api/v1/users/{user_id}`.

В `v1` ответы составляются непосредственно в обработчиках. Здесь нет схем запроса, схем ответа и хранилища. Эта версия нужна, чтобы отдельно показать `APIRouter`, префиксы и path-параметры.

### Версия 2

[`app/main_v2.py`](app/main_v2.py) подключает новые роутеры из каталога `api/v2` с префиксом `/api/v2`. Они не дописывают код `v1`, а образуют отдельный вариант приложения. Полный путь создания пользователя: `/api/v2` + `/users` + пустой суффикс обработчика = `POST /api/v2/users`.

[`app/models/schemas.py`](app/models/schemas.py) задаёт входные и выходные данные. `UserCreate` проверяет тело запроса, а `UserResponse` определяет ответ и не содержит пароль. Роутеры `v2` используют временные списки `fake_users_db` и `fake_items_db`. Данные существуют только во время работы процесса. `HTTPException` возвращает `404`, если пользователя нет. Эта версия показывает валидацию, response model, коды ответа и обработку ожидаемых ошибок.

### Версия 3

[`app/main_v3.py`](app/main_v3.py) подключает роутер текстового endpoint с префиксом `/api/v3` и отдельно объявляет `/health`. В [`app/api/v3/llm.py`](app/api/v3/llm.py) обработчик добавляет `/predict-text`, поэтому полный адрес равен `POST /api/v3/predict-text`. Для тела запроса и ответа используются `TextPredictIn` и `TextPredictOut` из `schemas.py`.

`v3` не содержит маршруты пользователей и предметов из `v2`. Это отдельный учебный пример API для ML-сервиса, а не полное продолжение предыдущего приложения. Настоящая модель здесь не вызывается. Версия нужна, чтобы показать контракт inference endpoint, `/health`, версию модели и измерение времени обработки.


## 1. Подготовка

Откройте терминал в директории `practice_6` и установите зависимости:

```bash
python -m pip install -r requirements.txt
```

Notebook также нужно запускать из директории `practice_6`. Тогда Python найдёт пакет `app`, расположенный в `practice_6/app`.

В следующей ячейке каждое приложение импортируется под коротким именем.
Например, `from app.main_v1 import app as v1_app` означает: взять объект `app` из файла `app/main_v1.py` и внутри notebook обращаться к нему как к `v1_app`.

В ячейке подготовки списки `fake_users_db` и `fake_items_db` очищаются. Благодаря этому повторный запуск notebook начинается с пустого состояния, а первый созданный объект снова получает `id=1`.


In [1]:
import fastapi
import pydantic
from fastapi.testclient import TestClient

from app.api.v2 import items, users
from app.main import app as basic_app
from app.main_v1 import app as v1_app
from app.main_v2 import app as v2_app
from app.main_v3 import app as v3_app

users.fake_users_db.clear()
items.fake_items_db.clear()

print("FastAPI:", fastapi.__version__)
print("Pydantic:", pydantic.__version__)


FastAPI: 0.162.0
Pydantic: 2.14.1


## 2. Базовые маршруты

`app.main` содержит корневой маршрут, path-параметр и query-параметры. Декоратор `@app.get("/")` связывает запрос `GET /` с функцией, расположенной сразу под декоратором.

В адресе `/items/{item_id}` часть `{item_id}` является path-параметром. Запрос `/items/3` передаст число `3` в аргумент `item_id`.

Маршрут `/search` принимает два query-параметра:

- `query: str` содержит обязательный текст для поиска;
- `limit: int` содержит максимальное количество результатов;
- `default=10` задаёт значение `limit`, если его нет в запросе;
- `ge=1` требует значение не меньше `1`;
- `le=100` требует значение не больше `100`.

Запрос `/search?query=model&limit=5` передаст строку `model` в `query` и число `5` в `limit`. В `TestClient` словарь `params={"query": "model", "limit": 5}` преобразуется в эту часть URL. Если `limit` не передать, используется `10`. Если не передать обязательный `query` или указать `limit=0`, FastAPI вернёт `422` до вызова обработчика.


In [2]:
basic_client = TestClient(basic_app)

root_response = basic_client.get("/")
item_response = basic_client.get("/items/3")
invalid_item_response = basic_client.get("/items/not-a-number")
search_response = basic_client.get(
    "/search",
    params={"query": "model", "limit": 5},
)
invalid_limit_response = basic_client.get(
    "/search",
    params={"query": "model", "limit": 0},
)
default_limit_response = basic_client.get(
    "/search",
    params={"query": "model"},
)
missing_query_response = basic_client.get(
    "/search",
    params={"limit": 5},
)

print("root:", root_response.status_code, root_response.json())
print("item:", item_response.status_code, item_response.json())
print("invalid item ID:", invalid_item_response.status_code)
print("search:", search_response.status_code, search_response.json())
print("invalid limit:", invalid_limit_response.status_code)
print("default limit:", default_limit_response.status_code, default_limit_response.json())
print("missing query:", missing_query_response.status_code)


root: 200 {'message': 'Hello World'}
item: 200 {'item_id': 3}
invalid item ID: 422
search: 200 {'query': 'model', 'limit': 5}
invalid limit: 422
default limit: 200 {'query': 'model', 'limit': 10}
missing query: 422


## 3. Роутеры и учебная версия 1

Приложение создаётся в [`app/main_v1.py`](app/main_v1.py). Маршруты пользователей находятся в [`app/api/v1/users.py`](app/api/v1/users.py), а маршруты предметов в [`app/api/v1/items.py`](app/api/v1/items.py).

Префикс `/api/v1` задаётся при подключении роутеров. Префиксы `/users` и `/items` задаются внутри самих роутеров. Поэтому получаются адреса `/api/v1/users` и `/api/v1/items`.

Данные `Alice` и `Book` записаны непосредственно в функциях и не сохраняются в базе. Маршрут `/users` без `/api/v1` не зарегистрирован, поэтому запрос к нему возвращает `404`.


In [3]:
v1_client = TestClient(v1_app)

v1_users = v1_client.get("/api/v1/users")
v1_items = v1_client.get("/api/v1/items")
unversioned_users = v1_client.get("/users")

print("users:", v1_users.status_code, v1_users.json())
print("items:", v1_items.status_code, v1_items.json())
print("unversioned path:", unversioned_users.status_code)


users: 200 [{'id': 1, 'name': 'Alice'}]
items: 200 [{'id': 1, 'title': 'Book'}]
unversioned path: 404


## 4. Валидация и учебная версия 2

Приложение создаётся в [`app/main_v2.py`](app/main_v2.py). Обработчики находятся в [`app/api/v2/users.py`](app/api/v2/users.py) и [`app/api/v2/items.py`](app/api/v2/items.py), а Pydantic-модели описаны в [`app/models/schemas.py`](app/models/schemas.py).

Аргумент `json=user_payload` передаёт словарь как JSON-тело запроса. `UserCreate` проверяет поля `name`, `email` и `password`. `status_code=201` означает, что пользователь создан. `response_model=UserResponse` проверяет ответ и не позволяет вернуть пароль клиенту.

Созданные пользователи хранятся в списке `fake_users_db` только до завершения Python-процесса. Это демонстрационное хранилище, а не база данных.


In [4]:
v2_client = TestClient(v2_app)

user_payload = {
    "name": "Alice",
    "email": "alice@example.com",
    "password": "secret1",
}
created_user = v2_client.post("/api/v2/users", json=user_payload)

print("status:", created_user.status_code)
print("response:", created_user.json())
print("password returned:", "password" in created_user.json())


status: 201
response: {'name': 'Alice', 'email': 'alice@example.com', 'id': 1, 'items': []}
password returned: False


### Ошибки валидации и отсутствующий ресурс

Некорректный email возвращает `422`, потому что запрос не соответствует `UserCreate`. В ответе FastAPI указывает поле с ошибкой и причину отказа. Обработчик `create_user` при этом не выполняется.

Запрос `/api/v2/users/999` возвращает `404`, потому что в `fake_users_db` нет пользователя с идентификатором `999`. Поле `detail` содержит сообщение `User not found`.


In [5]:
invalid_email = v2_client.post(
    "/api/v2/users",
    json={
        "name": "Bob",
        "email": "invalid",
        "password": "secret1",
    },
)
missing_user = v2_client.get("/api/v2/users/999")

print("invalid email:", invalid_email.status_code, invalid_email.json())
print("missing user:", missing_user.status_code, missing_user.json())


invalid email: 422 {'detail': [{'type': 'value_error', 'loc': ['body', 'email'], 'msg': 'value is not a valid email address: An email address must have an @-sign.', 'input': 'invalid', 'ctx': {'reason': 'An email address must have an @-sign.'}}]}
missing user: 404 {'detail': 'User not found'}


### Создание и получение предметов

`ItemCreate` проверяет `title`, необязательное `description` и `owner_id`. Значение `owner_id=1` ссылается на пользователя, созданного в предыдущей ячейке.

Если владельца нет, API возвращает `404`. Если владелец найден, предмет сохраняется в `fake_items_db` и добавляется в поле `items` этого пользователя. `ItemResponse` определяет поля ответа.


In [6]:
item_payload = {
    "title": "Model card",
    "description": "Version 1",
    "owner_id": 1,
}
created_item = v2_client.post("/api/v2/items", json=item_payload)
item_list = v2_client.get("/api/v2/items")
owner_after_item = v2_client.get("/api/v2/users/1")
missing_owner = v2_client.post(
    "/api/v2/items",
    json={"title": "Orphan item", "owner_id": 999},
)

print("created:", created_item.status_code, created_item.json())
print("list:", item_list.status_code, item_list.json())
print("owner items:", owner_after_item.json()["items"])
print("missing owner:", missing_owner.status_code, missing_owner.json())


created: 201 {'title': 'Model card', 'description': 'Version 1', 'id': 1, 'owner_id': 1}
list: 200 [{'title': 'Model card', 'description': 'Version 1', 'id': 1, 'owner_id': 1}]
owner items: [{'title': 'Model card', 'description': 'Version 1', 'id': 1, 'owner_id': 1}]
missing owner: 404 {'detail': 'Owner not found'}


## 5. OpenAPI и документация

FastAPI создаёт OpenAPI-схему из маршрутов, типов параметров и `response_model`. JSON-схема доступна по `/openapi.json`, а Swagger UI по `/docs`.

Документация относится к конкретному объекту приложения. Здесь запросы выполняются через `v2_client`, поэтому в `paths` находятся только маршруты `v2`. При запуске `uvicorn app.main_v2:app` те же страницы можно открыть в браузере по `http://127.0.0.1:8000/docs` и `http://127.0.0.1:8000/openapi.json`.


In [7]:
docs_response = v2_client.get("/docs")
openapi_response = v2_client.get("/openapi.json")
openapi_paths = sorted(openapi_response.json()["paths"])

print("docs:", docs_response.status_code)
print("openapi:", openapi_response.status_code)
print("paths:", openapi_paths)


docs: 200
openapi: 200
paths: ['/api/v2/items', '/api/v2/users', '/api/v2/users/{user_id}']


## 6. Учебная версия 3 и заглушка модели

Приложение и маршрут `/health` находятся в [`app/main_v3.py`](app/main_v3.py). Обработчик `/api/v3/predict-text` находится в [`app/api/v3/llm.py`](app/api/v3/llm.py).

`POST /api/v3/predict-text` получает JSON `{"prompt": "Привет"}`. Модель `TextPredictIn` проверяет, что `prompt` является непустой строкой. Пустое значение возвращает `422` до вызова обработчика. Обработчик вручную формирует поле `reply`, возвращает фиксированное значение `model_version` и измеряет только время выполнения заглушки. Настоящая ML-модель не загружается и не вызывается.

Запрос к `/api/v2/predict-text` специально выполняется для проверки версии адреса. Такого маршрута в `v3_app` нет, поэтому ожидается `404`.


In [8]:
v3_client = TestClient(v3_app)

health_response = v3_client.get("/health")
prediction_response = v3_client.post(
    "/api/v3/predict-text",
    json={"prompt": "Привет"},
)
empty_prompt_response = v3_client.post(
    "/api/v3/predict-text",
    json={"prompt": ""},
)
old_version_response = v3_client.post(
    "/api/v2/predict-text",
    json={"prompt": "Привет"},
)

print("health:", health_response.status_code, health_response.json())
print("prediction:", prediction_response.status_code, prediction_response.json())
print("empty prompt:", empty_prompt_response.status_code)
print("old version:", old_version_response.status_code)


health: 200 {'status': 'ok', 'model_version': 'demo-llm-v1'}
prediction: 200 {'reply': 'Модель отвечает на запрос: Привет', 'model_version': 'demo-llm-v1', 'latency_ms': 0.001}
empty prompt: 422
old version: 404


## 7. Проверка контрактов

Каждый `assert` проверяет одно ожидаемое свойство ответа. Например, `assert created_user.status_code == 201` подтверждает код создания, а `assert "password" not in created_user.json()` подтверждает отсутствие пароля в JSON.

Если условие ложно, Python создаёт `AssertionError` и останавливает ячейку. Сообщение `Notebook checks passed.` появляется только после успешного выполнения всех проверок.


In [9]:
assert root_response.status_code == 200
assert item_response.json() == {"item_id": 3}
assert invalid_item_response.status_code == 422
assert default_limit_response.json()["limit"] == 10
assert missing_query_response.status_code == 422
assert invalid_limit_response.status_code == 422
assert unversioned_users.status_code == 404

assert created_user.status_code == 201
assert "password" not in created_user.json()
assert invalid_email.status_code == 422
assert missing_user.status_code == 404
assert missing_user.json() == {"detail": "User not found"}

assert created_item.status_code == 201
assert item_list.json() == [created_item.json()]
assert owner_after_item.json()["items"] == [created_item.json()]
assert missing_owner.status_code == 404
assert missing_owner.json() == {"detail": "Owner not found"}
assert docs_response.status_code == 200
assert openapi_response.status_code == 200

assert health_response.json()["status"] == "ok"
assert prediction_response.status_code == 200
assert prediction_response.json()["model_version"] == "demo-llm-v1"
assert empty_prompt_response.status_code == 422
assert old_version_response.status_code == 404

print("Notebook checks passed.")


Notebook checks passed.
